# 3D Taylor Açılımı — x₁x₂x₃ + sin(x₁)

Bu notebook, AOT paketini kullanarak üç değişkenli fonksiyonların 2. mertebe Taylor açılımını gösterir.

> **Not:** 3D fonksiyonlar için görselleştirme desteklenmez (4D gerekir). Sembolik ve sayısal sonuçlara erişilebilir.

In [ ]:
import sympy as sp
import numpy as np
from aot import TaylorExpansion

## 1. Fonksiyon Tanımı

In [ ]:
x1, x2, x3 = sp.symbols('x1 x2 x3')
f = x1*x2*x3 + sp.sin(x1)

T = TaylorExpansion(f=f, variables=[x1, x2, x3], point=[0, 0, 0], order=2)
print("Fonksiyon: x1*x2*x3 + sin(x1)")
print("Açılım noktası: (0, 0, 0)")

## 2. Sembolik Hesap

In [ ]:
print("Taylor açılımı:")
T  # LaTeX render

In [ ]:
print("Gradient (∇f):\n", T.gradient)
print("\nHessian (H):\n", T.hessian)

## 3. Sayısal Karşılaştırma

In [ ]:
import pandas as pd

f_num = T.to_numeric()
t_num = T.taylor_to_numeric()

test_points = [
    (0, 0, 0),
    (0.1, 0.1, 0.1),
    (0.2, 0.3, 0.1),
    (0.5, 0.5, 0.5),
]
rows = []
for x1v, x2v, x3v in test_points:
    fv = float(f_num(x1v, x2v, x3v))
    tv = float(t_num(x1v, x2v, x3v))
    rows.append({"x1": x1v, "x2": x2v, "x3": x3v,
                 "f(x)": fv, "T(x)": tv, "|f-T|": abs(fv - tv)})

pd.DataFrame(rows).round(6)

## 4. Görselleştirme

3D fonksiyonlar için `plot()` çağrısı `NotImplementedError` fırlatır.

In [ ]:
try:
    T.plot()
except NotImplementedError as e:
    print(f"Beklenen hata: {e}")

## 5. Hata Analizi (1D kesit)

x2=x3=0 sabit tutarak x1 ekseni boyunca hata analizi yapılır.

In [ ]:
import matplotlib.pyplot as plt

x1_vals = np.linspace(-1.5, 1.5, 300)
zeros = np.zeros_like(x1_vals)

error = np.abs(f_num(x1_vals, zeros, zeros) - t_num(x1_vals, zeros, zeros))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(x1_vals, error + 1e-16, color='purple')  # +1e-16: log(0) kaçınma
ax.set_xlabel('x1 (x2=x3=0)')
ax.set_ylabel('|f(x) - T(x)|')
ax.set_title('Hata Analizi: x1*x2*x3 + sin(x1) Taylor Açılımı')
ax.grid(True, alpha=0.3)
fig